In [1]:
import os
from urllib.request import urlretrieve
import geopandas as gpd
import zipfile
import pandas as pd

In [2]:
BASE_URL = "https://minedbuildings.z5.web.core.windows.net/legacy/usbuildings-v2/"
URL_EXT = ".geojson.zip"
STATES_CAISO = ["California", "Arizona", "Nevada"]
STATES_MAR23 = ['WestVirginia', 'Illinois', 'Minnesota', 'Wisconsin', 'Ohio', 'Texas', \
                'Oklahoma', 'Tennessee', 'Arkansas', 'Mississippi', 'Missouri', 'Kansas', 'Indiana', 'Iowa']

OUT_DIR = "../data/building_footprints"

urls_caiso = []
for i in range(len(STATES_CAISO)):
    urls_caiso.append(BASE_URL + STATES_CAISO[i] + URL_EXT)

urls_mar23 = []
for i in range(len(STATES_MAR23)):
    urls_mar23.append(BASE_URL + STATES_MAR23[i] + URL_EXT)

def pull_building_data(urls):

    os.makedirs(OUT_DIR, exist_ok=True)
    fnames = []
    for url in urls:
        basenm = url.split("/")[-1]
        print(basenm)
        filename = os.path.join(OUT_DIR, basenm)
        urlretrieve(url, filename)

        print(filename)
        with zipfile.ZipFile(filename,"r") as zip_ref:
            zip_ref.extractall(OUT_DIR)
            fnames.append(os.path.splitext(filename)[0])

pull_building_data(urls_caiso)
pull_building_data(urls_mar23)

In [3]:
def get_fnames(urls):
    fnames = []
    for i in range(len(urls)):
        fnames.append(os.path.join(OUT_DIR,os.path.splitext(urls[i].split("/")[-1])[0]))
    return fnames


fnames_caiso = get_fnames(urls_caiso)
fnames_mar23 = get_fnames(urls_mar23)
print(fnames_caiso)

['../data/building_footprints/California.geojson', '../data/building_footprints/Arizona.geojson', '../data/building_footprints/Nevada.geojson']


In [4]:

# Load NERC regions - Code taken from ./notebooks/California_shape_files.ipynb
regions_gdf = gpd.read_file("../data/NERC_Reliability_Coordinators.geojson")
# Ensure the CRS is WGS84 (EPSG:4326)
if regions_gdf.crs is None or regions_gdf.crs.to_string() != "EPSG:4326":
    print('converting to EPSG: 4326')
    regions_gdf = regions_gdf.to_crs("EPSG:4326")
CAISO = regions_gdf[regions_gdf['NAME']=='CALIFORNIA INDEPENDENT SYSTEM OPERATOR']

In [5]:
#Load ROI for March 2023 event
mar23_roi = gpd.read_file("../data/selected_states_MarchApril2023_event.geojson")
print(mar23_roi.keys(), len(mar23_roi))
if mar23_roi.crs is None or mar23_roi.crs.to_string() != "EPSG:4326":
    print('converting to EPSG: 4326')
    mar23_roi = mar23_roi.to_crs("EPSG:4326")

states = []
for i, row in mar23_roi.iterrows():
    states.append(row["NAME"])
print(states)

Index(['REGION', 'DIVISION', 'STATEFP', 'STATENS', 'GEOID', 'GEOIDFQ',
       'STUSPS', 'NAME', 'LSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object') 14
converting to EPSG: 4326
['West Virginia', 'Illinois', 'Minnesota', 'Wisconsin', 'Ohio', 'Texas', 'Oklahoma', 'Tennessee', 'Arkansas', 'Mississippi', 'Missouri', 'Kansas', 'Indiana', 'Iowa']


In [ ]:
def row_intersects(row, roi):
    intersects = False
    for idx, roi_row in roi.iterrows():
        intersects = intersects or row['geometry'].intersects(roi_row['geometry'])
        if intersects:
            break
    return intersects
    
def clip_buildings_to_roi(buildings_fnames, roi, intersection_key):
    #Read in individual state data for buildings 
    for i in range(len(buildings_fnames)):
        building_df = gpd.read_file(buildings_fnames[i])
        building_df.set_geometry('geometry',inplace=True)
        building_df = building_df.to_crs('EPSG:4326')
        print(building_df.keys())
 
        #Find buildings that intersect ROI
        building_df[intersection_key] = False
        building_df[intersection_key] = building_df.apply(row_intersects, roi=roi, axis=1)
        building_subset = building_df[building_df[intersection_key] == True]
        print(len(building_subset), len(building_df))

        out_fname = os.path.splitext(buildings_fnames[i])[0] + "_" + intersection_key  + ".parquet"
        building_subset.to_parquet(out_fname, index=True)

clip_buildings_to_roi(fnames_caiso, CAISO, 'caiso_intersections')
clip_buildings_to_roi(fnames_mar23, mar23_roi, 'mar23_event_roi_intersections')
    